# 07 — Research Synthesis, Publication Artifacts, and Interactive Dashboard Integration
### Stage 8: Paper Synthesis, Figure Generation, and Production Dashboard Integration

---

## Executive Overview

In this culminating notebook, we synthesize the complete empirical journey of the **Hindi Lexical Sensitivity** research project:
1. **Holistic Scientific Synthesis**: Connect findings across dataset integrity, subword fragmentation collapse, MLM plausibility, and explainability orthogonals.
2. **Publication-Ready Figures**: Generate all camera-ready figures for LaTeX/Typst paper submissions.
3. **Interactive Dashboard Data Pipeline**: Produce verified JSON manifests and tidy CSVs for the React/Vite dashboard (`dashboard/`).
4. **Reproducibility Audit & Practical Guidelines**: Provide concrete recommendations for multilingual NLP practitioners.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.analysis.dashboard_data import run_stage8_dashboard_data_preparation

print("Dashboard & synthesis modules loaded.")

Dashboard & synthesis modules loaded.


C:\Users\Friends\.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at C:\Users\Friends\AppData\Local\Temp\matplotlib-8mmqkou1 because there was an issue with the default path (C:\Users\Friends\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


---

## 1. Executive Scientific Synthesis: The Four Pillars of Findings

| Pillar | Empirical Finding | Scientific Mechanism |
|---|---|---|
| **1. Dataset Integrity** | 200 Devanagari pairs, 100% script adherence, non-destructive row preservation. | Human annotation disagreements and rejected items retained to prevent selection bias. |
| **2. Subword Collapse** | Single-subword coverage collapses by **$-50\%$ to $-60\%$** under Devanagari spelling errors. | Subword tokenizers fragment out-of-vocabulary matra length and ligature variations into character pieces. |
| **3. Model Plausibility** | Synonyms incur minimal penalty ($\Delta 	ext{MeanLogP} pprox 0$), while context errors and spelling errors drop sharply. | Pretrained MLMs preserve plausibility under meaning-preserving variants but strongly penalize statistical infrequency. |
| **4. Explainability Orthogonal** | Explanation stability does **not** correlate with faithfulness ($p > 0.05$ across all tests). | Stability under input noise and faithfulness to model mechanisms capture distinct, independent dimensions of explainability. |

---

## 2. Generating Publication Figures

Let's reproduce the master figures used in the scientific research paper and audit reports.

In [2]:
# Load merged analysis dataset
df_combined = pd.read_csv(ROOT_DIR / "outputs" / "analysis" / "combined_pair_level_results.csv")
df_rel = pd.read_csv(ROOT_DIR / "outputs" / "hindi_faithfulness" / "stability_faithfulness_relationship.csv")
df_faith_summary = pd.read_csv(ROOT_DIR / "outputs" / "hindi_faithfulness" / "faithfulness_summary.csv")

# Create figure directory
FIG_DIR = ROOT_DIR / "outputs" / "paper_analysis" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures directory: {FIG_DIR}")

Figures directory: C:\Users\Friends\Downloads\hindi-lexical-sensitivity (3)\hindi-lexical-sensitivity\outputs\paper_analysis\figures


In [3]:
# Publication Figure: Faithfulness Gap by Masking Fraction
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
models = df_faith_summary["model_name"].unique()

for idx, model in enumerate(models):
    ax = axes[idx]
    m_name = model.split("/")[-1]
    m_df = df_faith_summary[df_faith_summary["model_name"] == model]
    
    for method, color, marker in [("attention", "#2b5c8f", "o"), ("integrated_gradients", "#d95f02", "s")]:
        sub = m_df[m_df["method"] == method].sort_values("fraction")
        ax.plot(
            sub["fraction"] * 100,
            sub["faithfulness_gap_mean"],
            marker=marker,
            linewidth=2.5,
            markersize=8,
            color=color,
            label=f"{method.capitalize()}"
        )
    
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"{m_name} — Faithfulness Gap vs. Masking %", fontsize=12, fontweight="bold")
    ax.set_xlabel("Context Masking Fraction (%)", fontsize=11)
    ax.set_ylabel("Mean Faithfulness Gap (ΔS_top - ΔS_bottom)", fontsize=11)
    ax.set_xticks([10, 20, 30])
    ax.legend()
    ax.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_faithfulness_by_fraction.png", dpi=300)
plt.show()
print("Saved: fig_faithfulness_by_fraction.png")

Saved: fig_faithfulness_by_fraction.png


---

## 3. Interactive React/Vite Dashboard Data Integration

The project includes an interactive web dashboard in `dashboard/` built with React, Vite, and Lucide icons.
To power the dashboard without requiring a live Python backend, we generate verified, lightweight JSON and CSV data tables in `outputs/dashboard_data/`.

In [4]:
print("Generating export data for interactive dashboard...")
manifest = run_stage8_dashboard_data_preparation()

print("\nDashboard Data Manifest:")
print(json.dumps(manifest, indent=2))

Generating export data for interactive dashboard...

Dashboard Data Manifest:
{
  "dataset_sha256": "f9a5af81e21624de47ead7feabba412149939c1eb9934f8fc2b8d452a232cd44",
  "total_source_pairs": 200,
  "dashboard_files_generated": [
    "C:\\Users\\Friends\\Downloads\\hindi-lexical-sensitivity (3)\\hindi-lexical-sensitivity\\outputs\\dashboard_data\\dataset_overview.json",
    "C:\\Users\\Friends\\Downloads\\hindi-lexical-sensitivity (3)\\hindi-lexical-sensitivity\\outputs\\dashboard_data\\dataset_tracks.csv",
    "C:\\Users\\Friends\\Downloads\\hindi-lexical-sensitivity (3)\\hindi-lexical-sensitivity\\outputs\\dashboard_data\\tokenization_summary.csv",
    "C:\\Users\\Friends\\Downloads\\hindi-lexical-sensitivity (3)\\hindi-lexical-sensitivity\\outputs\\dashboard_data\\scoring_summary.csv",
    "C:\\Users\\Friends\\Downloads\\hindi-lexical-sensitivity (3)\\hindi-lexical-sensitivity\\outputs\\dashboard_data\\stability_summary.csv",
    "C:\\Users\\Friends\\Downloads\\hindi-lexical-sensiti

---

## 4. Practical Guidelines for Multilingual & Devanagari NLP

Based on our empirical findings, we recommend the following engineering best practices:

1. **Devanagari Normalization Preprocessing**:
   - Apply Unicode canonical normalization (`NFKC`) and Unicode anusvara/nasal standardizations before tokenization.
   - For educational or user-facing spelling feedback systems, train tokenizers on synthetic orthographic noise so that Devanagari misspellings are not catastrophically fragmented.

2. **Scoring Bidirectional Models**:
   - Always use length-normalized Pseudo-Log-Likelihood ($	ext{MeanLogP}$) rather than raw PLL sums when comparing sentences of unequal token lengths.
   - Never compare raw logit values across models with different vocabulary sizes.

3. **Interpretable NLP Auditing**:
   - Do **not** rely on explanation stability under input noise as evidence of explanation faithfulness.
   - For masked language models, **Self-Attention Rollup** produces significantly more faithful discrete token attributions than continuous Integrated Gradients.

---

## Project Conclusion

Congratulations! You have navigated the complete, end-to-end Machine Learning research pipeline for **Hindi Lexical Sensitivity**:
- **01**: Validated raw dataset integrity and Devanagari script adherence.
- **02**: Discovered and quantified the Devanagari subword fragmentation collapse.
- **03**: Scored bidirectional MLMs using length-normalized Pseudo-Log-Likelihood.
- **04**: Evaluated cross-model concordance and statistical fragmentation correlations.
- **05**: Implemented Attention and Integrated Gradients feature attribution.
- **06**: Resolved the Stage 6 artifact and proved the orthogonality of Stability and Faithfulness.
- **07**: Synthesized publication figures and exported data for interactive visualization.